## Generative Adversarial Network

This notebook provides a modernised implementation of the first GAN developed by [Goodfellow et al. (2014)](https://proceedings.neurips.cc/paper/2014/file/5ca3e9b122f61f8f06494c97b1afccf3-Paper.pdf).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import Adam
# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as F
# tqdm
from tqdm.notebook import tqdm, trange

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Load and Prepare MNIST Dataset
MNIST is a built in dataset with torchvision and we load it here and prepare dataloaders for PyTorch.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
mnist_trainset = MNIST(root='./data', train=True, download=True, transform=transform)

In [ ]:
train_batch_size = 128
train_loader = data.DataLoader(mnist_trainset, batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 28, 28), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Generator class
The generator class represents the Generator network. The network has a noise vector as input and outputs vectors of length 784 = 28 x 28 corresponding to new generator images that match MNIST. The network itself uses 4 blocks consisting of a dense layer followed by batch normalization and then ReLU activation. The final layer does not have batch normalization and concludes with a Sigmoid function that matches the image scaling to the range [0, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=10, out_dim=784, hidden_dim=128):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim * 2, hidden_dim * 4),
            nn.BatchNorm1d(hidden_dim * 4),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim * 4, hidden_dim * 8),
            nn.BatchNorm1d(hidden_dim * 8),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim * 8, out_dim),
            nn.Sigmoid(),
        )
        
    def forward(self, noise):
        return self.gen(noise)

### Build the Discriminator class
The discriminator class represents the Discriminator network that will be trained to determine if an image is generated or from the real data distribution. This is a standard classification network using three dense layers with Leaky ReLU activation (note the final Sigmoid is in the loss function).

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_dim=784, hidden_dim=128, neg_slope=0.2):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 4),
            nn.LeakyReLU(negative_slope=neg_slope, inplace=True),
            nn.Linear(hidden_dim * 4, hidden_dim * 2),
            nn.LeakyReLU(negative_slope=neg_slope, inplace=True),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(negative_slope=neg_slope, inplace=True),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, image):
        return self.disc(image)

### Train the Model
The loss functions must be defined first and then the training loop can be run.

In [ ]:
def gen_loss(gen, disc, batch_size, latent_dim, bce_loss, device):
    noise = torch.randn(batch_size, latent_dim, device=device)
    fake_im = gen(noise)
    fake_pred = disc(fake_im)
    gen_loss = bce_loss(fake_pred, torch.ones_like(fake_pred))
    return gen_loss

In [ ]:
def disc_loss(gen, disc, real_im, batch_size, latent_dim, bce_loss, device):
    noise = torch.randn(batch_size, latent_dim, device=device)
    fake_im = gen(noise).detach()
    fake_pred = disc(fake_im)
    fake_loss = bce_loss(fake_pred, torch.zeros_like(fake_pred))
    real_pred = disc(real_im)
    real_loss = bce_loss(real_pred, torch.ones_like(real_pred))
    disc_loss = 0.5 * (fake_loss + real_loss)
    return disc_loss

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
n_epochs = 200
latent_dim = 100
lr = 0.0002
beta_1 = 0.5
beta_2 = 0.999
epoch_show = 20
show_batch = 64

In [ ]:
gen = Generator(latent_dim).to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))

In [ ]:
gen_loss_lst = list()
disc_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_gen_loss = 0.0
    avg_disc_loss = 0.0
    num_items = 0
    for real_im, _ in tqdm(train_loader):
        cur_batch_size = len(real_im)
        real_im = real_im.view(cur_batch_size, -1).to(device)

        # Update discriminator
        disc_opt.zero_grad()
        dloss = disc_loss(gen, disc, real_im, cur_batch_size, 
                          latent_dim, bce_loss, device)
        dloss.backward(retain_graph=True)
        disc_opt.step()

        # Update Generator
        gen_opt.zero_grad()
        gloss = gen_loss(gen, disc, cur_batch_size, 
                         latent_dim, bce_loss, device)
        gloss.backward(retain_graph=True)
        gen_opt.step()

        avg_gen_loss += gloss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if epoch % epoch_show == 0:
        noise = torch.randn(show_batch, latent_dim, device=device)
        fake_im = gen(noise)
        show_tensor_images(fake_im)
        show_tensor_images(real_im)
        
    ave_gen_loss = avg_gen_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    gen_loss_lst.append(ave_gen_loss)
    disc_loss_lst.append(ave_disc_loss)
    tqdm_epoch.set_description('Ave Gen Loss: {:5f}, \
                               Ave Disc Loss: {:5f}'.format(ave_gen_loss, ave_disc_loss))
    torch.save(gen.state_dict(), 'genckpt.pth')
    torch.save(disc.state_dict(), 'discckpt.pth')

### Plot Loss

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
tqdm_epoch

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, gen_loss_lst, label='Generator Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('GANLosses.png', dpi=300, bbox_inches='tight')

### Plot Final Generated Images

In [ ]:
noise = torch.randn(40, latent_dim, device=device)
samples = gen(noise)
plt.figure(figsize=(10,20))
show_tensor_images(samples, 40, nrow=20, filename='GANGrid.png')